data ingestion- to load data from data sources such as api ,pdf

text loader

In [1]:
from langchain_community.document_loaders import TextLoader

In [2]:
loader=TextLoader("textdata.txt")
text=loader.load()
text

[Document(metadata={'source': 'textdata.txt'}, page_content='Data ingestion is the automated process of transporting, cleansing, and loading data from various sources (databases, APIs, files)\ninto a centralized repository, such as a data warehouse or lake, for analysis and storage. ')]

from pypdfloader

In [3]:
from langchain_community.document_loaders import PyPDFLoader
loade=PyPDFLoader("research_k.pdf")

In [4]:
# !pip install pypdf


In [5]:
docs=loade.load()
docs

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-12-06T12:36:38+05:30', 'author': 'khush', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-12-06T12:36:38+05:30', 'sourcemodified': "D:20251206123638+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'research_k.pdf', 'total_pages': 46, 'page': 0, 'page_label': '1'}, page_content='1\nTitle: Hybrid Deep Learning Architecture Integrating ConvNeXtV2, Swin\nTransformer, and CBAM for Enhanced Multiclass Brain Tumor Classification in\nMRI Images\nFirst Author Name: Khushvir Singh\nResearch Scholar, Department of Computer science and Engineering , I.K. Gujral\nPunjab Technical University, Jalandhar - Kapurthala Highway, VPO - Ibban,\nKapurthala-144603, Punjab, India\nE-mail:Khushvirsingh100@gmail.com\nPhone: 91-7658048680\nSecond Author Name:Dr. Pooja Sharma\nAssistant Proffesor,Department of Computer science and Engineering , I.K. Gujral\nPunjab Technical University, Jalandhar 

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
textsplit=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
document=textsplit.split_documents(docs)


In [7]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma,FAISS


def cleanpd(text):
    return text.encode("utf-8","ignore").decode("utf-8","ignore")
cleandoc=[]
for i in document:
    i.page_content=cleanpd(i.page_content)
    cleandoc.append(i)

In [8]:
emb=OllamaEmbeddings(model="qwen3-embedding:0.6b")
db=FAISS.from_documents(cleandoc,emb)

In [9]:
# !pip install faiss-cpu

In [10]:
retriever=db.as_retriever(search_type="similarity",search_kwargs={"k":5})

In [11]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
You are an expert research assistant.

Answer the question using ONLY the provided context.
If the answer is not in the context, say "Not found in the document."

Context:
{context}

Question:
{question}

Answer:
""")

In [12]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="gemma3:1b",temperature=0.3)



from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
def format_doc(docs):
    return "\n\n".join("doc.page_content for doc in docs")


rag_chain=({"context":retriever | format_doc,"question":RunnablePassthrough()}|prompt|llm|StrOutputParser())



In [13]:
query="What are the main objective of this paper?"
response=rag_chain.invoke(query)

In [14]:
print(response)

Not found in the document.
